In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
# import juliacall
import copy
from tqdm import tqdm
from pathlib import Path
import networkx as nx

from src.load_data import (
    read_graph_transport_networks_tntp,
    read_traffic_mat_transport_networks_tntp,
    read_graph_sndlib_xml,
    read_traffic_mat_sndlib_xml,
    scale_graph_bandwidth_and_cost
)

from src.shortest_paths_gt import get_graph_props

from src.models import BeckmannModel, TelecomModel
from src.algs import cyclic, ustm, frank_wolfe, frank_wolfe_gpu_cugraph, N_conjugate_frank_wolfe
from src.salim import SaddleOracle, combined_salim
from src.saddle_ta import salim_ta, chambolle_pock_ta
from src.approx import seq_quad, seq_quad_chp
from src.path_based import pb_gradproj_ta

import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator


plt.rcParams.update({'font.size': 13})
%config InlineBackend.figure_format = 'retina'

%matplotlib inline

# Load network data

In [3]:
networks_path = Path("./TransportationNetworks")
telecom_path = Path("./sndlib_xml")

if True:
    # folder = "SiouxFalls"
    # net_name = "SiouxFalls_net"
    # traffic_mat_name = "SiouxFalls_trips"
    # time_limit = 13

    folder = "Anaheim"
    net_name = "Anaheim_net"
    traffic_mat_name = "Anaheim_trips"
    time_limit = 38

    # folder = "Barcelona"
    # net_name = "Barcelona_net"
    # traffic_mat_name = "Barcelona_trips"
    # time_limit = 500

    # folder = "Austin"
    # net_name = "Austin_net"
    # traffic_mat_name = "Austin_trips_am"

    # folder = "Berlin-Friedrichshain"
    # net_name = "friedrichshain-center_net"
    # traffic_mat_name = "friedrichshain-center_trips"

    # folder = "Terrassa-Asymmetric"
    # net_name = "Terrassa-Asym_net"
    # traffic_mat_name = "Terrassa-Asym_trips"
    # time_limit = 260
    
    # folder = "Chicago-Sketch"
    # net_name = "ChicagoSketch_net"
    # traffic_mat_name = "ChicagoSketch_trips"

    # folder = "Berlin-Mitte-Center"
    # net_name = "berlin-mitte-center_net"
    # traffic_mat_name = "berlin-mitte-center_trips"
    # time_limit = 8
    
    # folder = "Berlin-Tiergarten"
    # net_name = "berlin-tiergarten_net"
    # traffic_mat_name = "berlin-tiergarten_trips"
    # time_limit = 6.5
    
    # folder = "Eastern-Massachusetts"
    # net_name = "EMA_net"
    # traffic_mat_name = "EMA_trips"
    # time_limit = 15


    net_file = networks_path / folder / f"{net_name}.tntp"
    traffic_mat_file = networks_path / folder / f"{traffic_mat_name}.tntp"
    graph, metadata = read_graph_transport_networks_tntp(net_file)
    # graph = scale_graph_bandwidth_and_cost(graph)
    correspondences = read_traffic_mat_transport_networks_tntp(traffic_mat_file, metadata)
    n = graph.number_of_nodes()
else:
    folder = "abilene"
    net_file = telecom_path / folder / f"{folder}.xml"
    graph = scale_graph_bandwidth_and_cost(read_graph_sndlib_xml(net_file))
    correspondences = read_traffic_mat_sndlib_xml(net_file, delim = 5)
    n = graph.number_of_nodes()


print(f"{graph.number_of_edges()=}, {graph.number_of_nodes()=}")

metadata["can_pass_through_zones"]=False
graph.number_of_edges()=914, graph.number_of_nodes()=454


In [4]:
traffic_mat = correspondences.traffic_mat.copy()
departures, arrivals = traffic_mat.sum(axis=1), traffic_mat.sum(axis=0)
l, w = departures, arrivals

In [5]:
1

1

# Create instances of models/oracles

In [6]:
mu_bm = None
model = "beckmann"
tie_break_eps = 1e-12

if model == "beckmann":
    beckmann_model = BeckmannModel(
        graph,
        copy.deepcopy(correspondences),
        mu_bm,
        shortest_paths_backend="cpu",
        shortest_paths_tie_break_eps=tie_break_eps,
    )
    beckmann_model_gpu = BeckmannModel(
        graph,
        copy.deepcopy(correspondences),
        mu_bm,
        shortest_paths_backend="cugraph",
        shortest_paths_tie_break_eps=tie_break_eps,
    )
    use_capacity = False
if model == "telecom":
    beckmann_model = TelecomModel(
        graph,
        copy.deepcopy(correspondences),
        mu_bm,
        shortest_paths_backend="cpu",
        shortest_paths_tie_break_eps=tie_break_eps,
    )
    beckmann_model_gpu = TelecomModel(
        graph,
        copy.deepcopy(correspondences),
        mu_bm,
        shortest_paths_backend="cugraph",
        shortest_paths_tie_break_eps=tie_break_eps,
    )
    use_capacity = True
# twostage_beckmann_model = TwostageModel(beckmann_model, departures=departures, arrivals=arrivals, gamma=0.1)
# saddle_oracle = SaddleOracle(twostage_beckmann_model.traffic_model, twostage_beckmann_model.gamma, l, w)

eps = 1e-6
mean_bw = beckmann_model.graph.ep.capacities.a.mean()
mean_cost = beckmann_model.graph.ep.free_flow_times.a.mean()
print(mean_bw, mean_cost)
# print(beckmann_model.graph.ep.bandwidths.a.mean(), beckmann_model.graph.ep.costs.a.mean())
# cost suboptimality <= eps * (average link cost * avg bandwidth * |E| \approx total cost when beta=1)
eps_abs = eps * mean_cost * mean_bw * graph.number_of_edges()

eps_cons_abs = eps * mean_bw
# sum of capacity violation <= eps * average link capacity
print(eps_abs, eps_cons_abs)
print("CPU shortest-path backend:", beckmann_model.shortest_paths_backend)
print("GPU requested backend resolved to:", beckmann_model_gpu.shortest_paths_backend)
print("tie_break_eps:", tie_break_eps)

6030.196936542669 0.8823533768661695
4.86317887193558 0.006030196936542669
CPU shortest-path backend: cpu
GPU requested backend resolved to: cugraph
tie_break_eps: 1e-12


In [7]:
print(np.count_nonzero(np.isfinite(beckmann_model.graph_props[1])))
print(beckmann_model.graph_props[1].shape)
print(np.all(beckmann_model.all_cases))

914
(914,)
False


In [8]:
A = incidence_mat = nx.incidence_matrix(beckmann_model.nx_graph, oriented=True).todense()

svals = np.linalg.svd(A)[1]
lam1 = svals[0]
lam2 = svals[svals > 1e-8][-1]
print(lam1, lam2)


3.9282442373540523 0.16274761956611836


# Obtain almost exact solution

In [9]:


A = incidence_mat = nx.incidence_matrix(beckmann_model.nx_graph, oriented=True).todense()

saddle_oracle = SaddleOracle(beckmann_model, None, None, None)

Ld = saddle_oracle.Bmul(beckmann_model.correspondences.traffic_mat).T
b = -Ld

print(A.shape, b.shape)

svals = np.linalg.svd(A)[1]
lam1 = svals[0] ** 2
lam2 = svals[svals > 1e-8][-1] ** 2

L_sq = lam1 + Ld.shape[0]

lam1 *= 2
lam2 /= 2
lam1, lam2

mu = 2e-3
L = 100
iters = 11

# if folder == "Terrassa-Asymmetric":
#     flows_true, _, \
#         _,_ = \
#         seq_quad(beckmann_model, iters=14, log_max_diff=False, need_log=True, use_capacity=use_capacity)
# elif folder == "SiouxFalls":
#     _, flows_true, _, _ = N_conjugate_frank_wolfe(
#         beckmann_model, eps_abs, max_iter=8000, stop_by_crit=True, cnt_conjugates=3, linesearch=True, log_max_diff=False, log_period=0)
# else:
#     flows_true, _ = pb_gradproj_ta(beckmann_model, iters=600, log_max_diff=False, log_period=-1, use_capacity=use_capacity)


(454, 914) (454, 38)


In [10]:
# np.save(f"experiments_data/true_flows_{folder}.npy", flows_true)
flows_true = np.load(f"experiments_data/true_flows_{folder}.npy")

# Run OFAC

In [11]:
data = {"fw": {}, "fw_gpu": {}, "nfw": {}, "nfw_gpu": {}, "chp": {}, "seq_quad": {}, "pb_gradproj": {}}
saddle_methods = ["apdhg", "chp"]
log_max_diff = False

base_times = beckmann_model.graph.ep.free_flow_times.a.copy()
cpu_flows, cpu_dist = beckmann_model.flows_on_shortest(base_times, return_distance_mat=True)
gpu_flows, gpu_dist = beckmann_model_gpu.flows_on_shortest(base_times, return_distance_mat=True)
print("distance_mat max abs diff:", float(np.max(np.abs(cpu_dist - gpu_dist))))
print("flows_on_shortest l2 diff:", float(np.linalg.norm(cpu_flows - gpu_flows)))

distance_mat max abs diff: 0.0
flows_on_shortest l2 diff: 0.0


In [12]:
# model_name = "ustm"
# data[model_name]["times"], data[model_name]["flows"], logs, _ = ustm(beckmann_model, eps_abs, max_iter=2000, stop_by_crit=False, solution_flows=flows_true)
# data[model_name]["dgap"], data[model_name]["cons"], data[model_name]["time_log"], data[model_name]["flows_diff"] = logs

In [14]:
model_name = "fw"
max_iter = 600
data[model_name]["times"], data[model_name]["flows"], logs, _ = frank_wolfe(
    beckmann_model,
    eps_abs,
    max_iter=max_iter,
    stop_by_crit=False,
    linesearch=True,
    log_max_diff=log_max_diff,
    log_period=max_iter // 400,
    solution_flows=flows_true,
    time_limit=time_limit,
)
data[model_name]["dgap"], data[model_name]["time_log"], data[model_name]["primal"], data[model_name]["r_gap"], data[model_name]["flows_diff"] = logs

model_name = "fw_gpu"
data[model_name]["times"], data[model_name]["flows"], logs, _ = frank_wolfe_gpu_cugraph(
    beckmann_model_gpu,
    eps_abs,
    max_iter=max_iter,
    stop_by_crit=False,
    log_max_diff=log_max_diff,
    log_period=max_iter // 400,
    solution_flows=flows_true,
    time_limit=time_limit,
)
data[model_name]["dgap"], data[model_name]["time_log"], data[model_name]["primal"], data[model_name]["r_gap"], data[model_name]["flows_diff"] = logs

  1%|▏         | 8/600 [00:23<28:38,  2.90s/it]


KeyboardInterrupt: 

In [ ]:
# model_name = "nfw"
# max_iter = 6000
# data[model_name]["times"], data[model_name]["flows"], logs, _ = N_conjugate_frank_wolfe(
#     beckmann_model,
#     eps_abs,
#     max_iter=max_iter,
#     stop_by_crit=False,
#     cnt_conjugates=3,
#     linesearch=True,
#     log_max_diff=log_max_diff,
#     log_period=max_iter // 400,
#     solution_flows=flows_true,
#     time_limit=time_limit,
# )
# data[model_name]["dgap"], data[model_name]["time_log"], data[model_name]["primal"], data[model_name]["r_gap"], data[model_name]["flows_diff"] = logs

model_name = "nfw_gpu"
data[model_name]["times"], data[model_name]["flows"], logs, _ = N_conjugate_frank_wolfe(
    beckmann_model_gpu,
    eps_abs,
    max_iter=max_iter,
    stop_by_crit=False,
    cnt_conjugates=3,
    linesearch=True,
    log_max_diff=log_max_diff,
    log_period=max_iter // 400,
    solution_flows=flows_true,
    time_limit=time_limit,
)
data[model_name]["dgap"], data[model_name]["time_log"], data[model_name]["primal"], data[model_name]["r_gap"], data[model_name]["flows_diff"] = logs

cpu_total_time = data["nfw"]["time_log"][-1] if data["nfw"]["time_log"] else np.nan
gpu_total_time = data["nfw_gpu"]["time_log"][-1] if data["nfw_gpu"]["time_log"] else np.nan
if np.isfinite(cpu_total_time) and np.isfinite(gpu_total_time) and gpu_total_time > 0:
    print("NFW speedup CPU/GPU:", float(cpu_total_time / gpu_total_time))

In [ ]:
# A = incidence_mat = nx.incidence_matrix(beckmann_model.nx_graph, oriented=True).todense()

# saddle_oracle = SaddleOracle(beckmann_model, None, None, None)

# Ld = saddle_oracle.Bmul(beckmann_model.correspondences.traffic_mat).T
# b = -Ld

# svals = np.linalg.svd(A)[1]
# lam1 = svals[0] ** 2
# lam2 = svals[svals > 1e-8][-1] ** 2

# L_sq = lam1 + Ld.shape[0]

# lam1 *= 2
# lam2 /= 2
# lam1, lam2

# mu = 1e-2
# L = 100
# iters = 100000

# model_name = "salim"
# data[model_name] = {}
# data[model_name]["flows"], data[model_name]["flows_diff_delta"], data[model_name]["cons"], data[model_name]["opt_log"], \
#     data[model_name]["time_log"], data[model_name]["primal"] , data[model_name]["flows_diff"] = \
#     salim_ta(beckmann_model, iters=iters, mu=mu, L=L, lam1=lam1, lam2=lam2, log_max_diff=log_max_diff, log_period=iters//400, solution_flows=flows_true)

In [ ]:
# iters = 190000
# restart = 30

# model_name = "apdhg"
# data[model_name] = {}
# data[model_name]["flows"], data[model_name]["flows_diff_delta"], data[model_name]["cons"], \
#     data[model_name]["opt_log"], data[model_name]["time_log"], data[model_name]["prox_n_iter"], \
#     data[model_name]["primal"], data[model_name]["flows_diff"] = \
#     apdhg(beckmann_model, iters=iters, restart=restart, log_max_diff=log_max_diff, log_period=iters//400, solution_flows=flows_true)

In [ ]:
beckmann_model.use_torch = True
beckmann_model.graph_props = get_graph_props(beckmann_model.graph, use_torch = True)

iters = 25000
restart = 30

model_name = "chp"
data[model_name] = {}
data[model_name]["flows"], data[model_name]["flows_diff_delta"], data[model_name]["cons"], \
    data[model_name]["opt_log"], data[model_name]["time_log"], \
    data[model_name]["primal"], _, data[model_name]["flows_diff"] = \
    chambolle_pock_ta(beckmann_model, iters=iters, log_max_diff=log_max_diff, log_period=iters//400, solution_flows=flows_true, time_limit=time_limit)

In [ ]:
# import cvxpy as cp
# print(cp.installed_solvers())

In [ ]:
# print(A.shape)

In [ ]:
# from juliacall import Main as jl

# # Инициализация CUDA и создание массива на GPU
# jl.seval('using Pkg; Pkg.status()')

In [ ]:
# iters = 10

# model_name = "seq_quad_ip"
# data[model_name] = {}
# data[model_name]["flows"], data[model_name]["cons"], data[model_name]["time_log"], \
#     data[model_name]["primal"], data[model_name]["flows_diff"] = \
#     seq_quad_ip(beckmann_model, iters=iters, log_max_diff=log_max_diff, need_log=True, solution_flows=flows_true)

In [ ]:
# iters = 10

# model_name = "seq_quad_chp"
# data[model_name] = {}
# data[model_name]["flows"], data[model_name]["cons"], data[model_name]["time_log"], \
#     data[model_name]["primal"], data[model_name]["flows_diff"] = \
#     seq_quad_chp(beckmann_model, iters=iters, log_max_diff=log_max_diff, need_log=True, solution_flows=flows_true, iters_quad=40000)

In [ ]:
model_name = "pb_gradproj"
max_iter=400
data[model_name]["flows"], data[model_name]["iters"], data[model_name]["time_log"], data[model_name]["primal"], _, data[model_name]["flows_diff"]\
    = pb_gradproj_ta(beckmann_model, iters=max_iter, log_max_diff=log_max_diff, log_period=max_iter//100, solution_flows=flows_true, use_capacity=use_capacity, time_limit=time_limit)


# Save results

In [ ]:
import json

In [ ]:
data["pb_gradproj"]['flows'] = data["pb_gradproj"]['flows'].tolist()

In [ ]:
# del data["seq_quad"]

In [ ]:
import json
from pathlib import Path


dirname = folder
Path(f"experiments_data/{dirname}").mkdir(parents=True, exist_ok=True)
with open(f"experiments_data/{dirname}/{dirname}.json", "w+") as fp:
    json.dump(data, fp, indent=4)
    

# Plot results

In [ ]:
display_names = {
    "fw": "Frank-Wolfe (CPU)",
    "fw_gpu": "Frank-Wolfe (GPU)",
    "nfw": "N-conjugate FW (CPU)",
    "nfw_gpu": "N-conjugate FW (GPU SP)",
    "chp": "Chambolle-Pock",
    "seq_quad": "Quad. Programming",
    "pb_gradproj": "Path-based"
}

In [ ]:
dirname = folder
with open(f"experiments_data/{dirname}/{dirname}.json", "r") as fp:
    data: dict = json.load(fp)

In [ ]:

    
    
plt.figure(figsize=(8,5))

true_fl = data["nfw"]["flows"]

for model_name, vals in data.items():
    print(model_name)
    time_list = vals["time_log"]
        
    if not "quad" in model_name:
        n = len(time_list) // 20
    else:
        n = 1
        
    time_log = np.array(time_list[::n])
                
    plt.plot(time_log, np.array(vals["flows_diff"][::n]) + 1e-14, label=display_names[model_name])
    plt.scatter(time_log, np.array(vals["flows_diff"][::n]) + 1e-14)

plt.minorticks_on()
plt.yscale("log")
# plt.ylim(3* 10**(6), 2*10**5)
plt.ylabel("Distance to the solution")
plt.xlabel("Runtime, seconds")

minor_locator = LogLocator(base=10.0, subs=np.arange(1.0, 10.0) * 0.1, numticks=12)
plt.gca().yaxis.set_minor_locator(minor_locator)

plt.grid(which='major', linestyle='-', linewidth='0.5', alpha=0.5)
plt.grid(which='minor', linestyle=':', linewidth='0.5', alpha=1) 

plt.legend(loc = (0.6, 0.2))

plt.savefig(f"experiments_data/{dirname}/{folder}_flow_dist.pdf", bbox_inches="tight")

In [ ]:
plt.clf()
plt.figure(figsize=(8,6))

cons_sum = data["nfw"]["primal"][-1]
for model_name, vals in data.items():
    if model_name == "chp":
        continue
    if min(vals["primal"]) < cons_sum:
        cons_sum = min(vals["primal"])

cons_sum -= 0.5 * 10**(-8)

for model_name, vals in data.items():
    time_list = vals["time_log"]
        
    if not "quad" in model_name:
        n = len(time_list) // 20
    else:
        n = 1
        
    if model_name[:5] == "salim":
        time_log = np.array(time_list[::n]) # / 10
    else:
        time_log = np.array(time_list[::n])
        
    prime = abs(np.array(vals["primal"][::n]) - cons_sum)
    plt.plot(time_log, prime, label=model_name)
    plt.scatter(time_log, prime)

plt.minorticks_on()
# plt.ylim(bottom=np.abs(np.array(data["seq_quad"]["primal"][-1]) - cons_sum) / cons_sum + 1e-20)
# plt.ylim(2*10**(-2), 0.4)
plt.yscale("log")
plt.ylabel("Difference with optimal value")
plt.xlabel("Runtime, seconds")

minor_locator = LogLocator(base=10.0, subs=np.arange(1.0, 10.0) * 0.1, numticks=12)
plt.gca().yaxis.set_minor_locator(minor_locator)

plt.grid(which='major', linestyle='-', linewidth='0.5', alpha=0.5)
plt.grid(which='minor', linestyle=':', linewidth='0.5', alpha=1) 

plt.legend()

plt.savefig(f"experiments_data/{dirname}/{folder}_primal.pdf", bbox_inches="tight")